In [275]:
import pandas as pd
import numpy as np
import requests
import io

In [276]:
url = "https://storage.googleapis.com/aus-road-deaths-data-engineering-project/bitre_fatalities_feb2026.csv"
response = requests.get(url)
df = pd.read_csv(io.StringIO(response.text), sep = ',', header = 4)

In [277]:
df.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,Road User,Gender,Age,National Remoteness Areas 2021,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period
0,5201106170079,WA,6,2011,Friday,22:30:00,Single,No,No,No,70,Driver,Male,46,Unknown,Unknown,Unknown,Unknown,No,No
1,4201105140046,SA,5,2011,Saturday,14:11:00,Single,No,No,No,100,Driver,Female,19,Unknown,Unknown,Unknown,Unknown,No,No
2,1199410070402,NSW,10,1994,Friday,18:00:00,Multiple,No,-9,No,60,Driver,Female,35,Unknown,Unknown,Unknown,Unknown,No,No
3,3201212280255,QLD,12,2012,Friday,21:00:00,Multiple,No,No,No,100,Passenger,Female,38,Unknown,Unknown,Unknown,Unknown,Yes,No
4,3201110220178,QLD,10,2011,Saturday,13:00:00,Multiple,No,No,No,60,Motorcycle rider,Male,28,Unknown,Unknown,Unknown,Unknown,No,No


In [278]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58389 entries, 0 to 58388
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Crash ID                        58389 non-null  int64 
 1   State                           58389 non-null  object
 2   Month                           58389 non-null  int64 
 3   Year                            58389 non-null  int64 
 4   Dayweek                         58389 non-null  object
 5   Time                            58389 non-null  object
 6   Crash Type                      58389 non-null  object
 7   Bus Involvement                 58389 non-null  object
 8   Heavy Rigid Truck Involvement   58389 non-null  object
 9   Articulated Truck Involvement   58389 non-null  object
 10  Speed Limit                     58389 non-null  int64 
 11  Road User                       58389 non-null  object
 12  Gender                          58389 non-null

## Transformation

**1. Convert Time into datetime type**

The format of Time is Object, I will change it to DateTime format. There is an issue where it appears to have "99:99:99" in Time's column, I will treat it as unknown times and turn it into NaT (Not a Time)

In [279]:
df['Time'] = df['Time'].str.strip()
df['Time'] = pd.to_datetime(df['Time'], format = '%H:%M:%S', errors = 'coerce').dt.time

In [280]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58389 entries, 0 to 58388
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Crash ID                        58389 non-null  int64 
 1   State                           58389 non-null  object
 2   Month                           58389 non-null  int64 
 3   Year                            58389 non-null  int64 
 4   Dayweek                         58389 non-null  object
 5   Time                            58348 non-null  object
 6   Crash Type                      58389 non-null  object
 7   Bus Involvement                 58389 non-null  object
 8   Heavy Rigid Truck Involvement   58389 non-null  object
 9   Articulated Truck Involvement   58389 non-null  object
 10  Speed Limit                     58389 non-null  int64 
 11  Road User                       58389 non-null  object
 12  Gender                          58389 non-null

**2. Handle the '-9' as missing/unknown values**

Change '-9' in 'Involvement' columns to 'Unknown'

In [281]:
involvement_cols = ['Bus Involvement', 
                    'Heavy Rigid Truck Involvement', 
                    'Articulated Truck Involvement']

for col in involvement_cols:
    #Convert to string first to ensure matching
    df[col] = df[col].str.strip()
    #Replace -9 with Unknown
    df[col] = df[col].replace('-9', 'Unknown')

Change '-9' in 'Age' column to NaN

In [282]:
number_cols = ['Age', 'Speed Limit'] 

for col in number_cols:
    df[col] = df[col].replace(-9, np.nan)

**3. Remove duplicate**

I only remove rows where every single column is the same

In [283]:
df = df.drop_duplicates()

**4. Create a "Holiday Flag" column**

This column uses Yes/No as values to indicate if the crash happened on holiday

In [284]:
df['Holiday Flag'] = df.apply(lambda x : 'Y'
                              if str(x['Christmas Period']).lower() == 'yes'
                              or str(x['Easter Period']).lower() == 'yes'
                              else 'N', axis = 1)

In [285]:
df['Weekend Flag'] = df.apply(lambda x: 'Y' 
                              if str(x['Dayweek']).lower() == 'saturday'
                              or str(x['Dayweek']).lower() == 'sunday'
                              else 'N', axis = 1)

## Data Modeling

**1. Create dim_date table**

In [296]:
dim_date = df[['Month', 'Year', 'Time', 'Holiday Flag', 'Weekend Flag']].drop_duplicates().reset_index(drop = True)

# Create a new column 'Date ID' as index
date_id = dim_date.index

dim_date.insert(loc = 0, column = 'Date ID', value = date_id)
dim_date.head()

,Date ID,Month,Year,Time,Holiday Flag,Weekend Flag
0,0,6,2011,22:30:00,N,N
1,1,5,2011,14:11:00,N,Y
2,2,10,1994,18:00:00,N,N
3,3,12,2012,21:00:00,Y,N
4,4,10,2011,13:00:00,N,Y


**2. Create dim_location table**

In [287]:
dim_location = df[['State', 'National Road Type', 'National Remoteness Areas 2021', 'SA4 Name 2021', 'National LGA Name 2021']].drop_duplicates().reset_index(drop = True)

# Create a new column 'Location ID' as index
location_id = dim_location.index

dim_location.insert(loc = 0, column = 'Location ID', value = location_id)

**3. Create dim_person table**

In [288]:
dim_person = df[['Gender', 'Age', 'Road User']].drop_duplicates().reset_index(drop = True)

# Create a new column 'Person ID' as index
person_id = dim_person.index

dim_person.insert(loc = 0, column = 'Person ID', value = person_id)
dim_person.head()

,Person ID,Gender,Age,Road User
0,0,Male,46.0,Driver
1,1,Female,19.0,Driver
2,2,Female,35.0,Driver
3,3,Female,38.0,Passenger
4,4,Male,28.0,Motorcycle rider


**4. Create dim_scenario table**

In [289]:
dim_scenario = df[['Crash Type', 'Speed Limit', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement']].drop_duplicates().reset_index(drop = True)

# Create a new column 'Scenario ID' as index
scenario_id = dim_scenario.index

dim_scenario.insert(loc = 0, column = 'Scenario ID', value = scenario_id)
dim_scenario.head()

,Scenario ID,Crash Type,Speed Limit,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement
0,0,Single,70.0,No,No,No
1,1,Single,100.0,No,No,No
2,2,Multiple,60.0,No,Unknown,No
3,3,Multiple,100.0,No,No,No
4,4,Multiple,60.0,No,No,No


**5. Create fact_fatalities table**

In [290]:
fact_fatalities = df.merge(dim_date, on = ['Month', 'Year', 'Time', 'Holiday Flag', 'Weekend Flag'], how = 'left')
fact_fatalities = fact_fatalities.merge(dim_location, on = ['State', 'National Road Type', 'National Remoteness Areas 2021', 'SA4 Name 2021', 'National LGA Name 2021'], how = 'left') 
fact_fatalities = fact_fatalities.merge(dim_person, on = ['Gender', 'Age', 'Road User'], how = 'left') 
fact_fatalities = fact_fatalities.merge(dim_scenario, on = ['Crash Type', 'Speed Limit', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement'], how = 'left')

In [292]:
fact_fatalities.head(100)

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,...,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Holiday Flag,Weekend Flag,Date ID,Location ID,Person ID,Scenario ID
0,5201106170079,WA,6,2011,Friday,22:30:00,Single,No,No,No,...,Unknown,Unknown,No,No,N,N,0,0,0,0
1,4201105140046,SA,5,2011,Saturday,14:11:00,Single,No,No,No,...,Unknown,Unknown,No,No,N,Y,1,1,1,1
2,1199410070402,NSW,10,1994,Friday,18:00:00,Multiple,No,Unknown,No,...,Unknown,Unknown,No,No,N,N,2,2,2,2
3,3201212280255,QLD,12,2012,Friday,21:00:00,Multiple,No,No,No,...,Unknown,Unknown,Yes,No,Y,N,3,3,3,3
4,3201110220178,QLD,10,2011,Saturday,13:00:00,Multiple,No,No,No,...,Unknown,Unknown,No,No,N,Y,4,3,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2200504270112,VIC,4,2005,Wednesday,19:25:00,Single,No,No,No,...,Unknown,Unknown,No,No,N,N,95,4,83,1
96,6200102180008,TAS,2,2001,Sunday,22:15:00,Single,Yes,Unknown,No,...,Unknown,Unknown,No,No,N,Y,96,17,84,30
97,1199706050241,NSW,6,1997,Thursday,05:08:00,Single,No,Unknown,No,...,Unknown,Unknown,No,No,N,N,97,2,85,12
98,3199001160007,QLD,1,1990,Tuesday,14:00:00,Multiple,No,Unknown,No,...,Unknown,Unknown,No,No,N,N,98,3,86,2


In [297]:
fact_fatalities_final = fact_fatalities[['Crash ID', 'Date ID', 'Location ID', 'Person ID', 'Scenario ID']]
fact_fatalities_final.head(100)

,Crash ID,Date ID,Location ID,Person ID,Scenario ID
0,5201106170079,0,0,0,0
1,4201105140046,1,1,1,1
2,1199410070402,2,2,2,2
3,3201212280255,3,3,3,3
4,3201110220178,4,3,4,4
...,...,...,...,...,...
95,2200504270112,95,4,83,1
96,6200102180008,96,17,84,30
97,1199706050241,97,2,85,12
98,3199001160007,98,3,86,2
